In [ ]:
import sys
sys.path.append("/path/to/example/autocompute/static")
from g16_env import load_gaussian_env
load_gaussian_env()

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw
from more_itertools import chunked
import matplotlib.pyplot as plt
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import SVG
import networkx as nx
import pandas as pd
from rdkit.Chem.rdchem import RWMol
from rdkit.Chem import rdmolops
from rdkit.Chem.Draw import IPythonConsole
import seaborn as sns
import re
from rdkit.Chem import AddHs
from rdkit.Chem import AllChem
import os
from openbabel import openbabel, pybel
from rdkit.Chem.rdmolfiles import MolToSmiles
from rdkit.Chem import RemoveHs
import subprocess
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import openbabel
from openpyxl import Workbook
import time
from openbabel import pybel
import shutil
import glob
from IPython.display import display
from rdkit.Chem import Draw
from openbabel import openbabel as ob

In [ ]:
import os
os.environ['PATH'] += ':/path/to/example/static/Multiwfn_3.8_dev_bin_Linux'

In [ ]:
IPythonConsole.ipython_useSVG = True

In [ ]:

def repeat_unit_charge_list(chg_filename):
    
    repeat_unit_charges = []
    with open(chg_filename, 'r') as file:
        for line in file:
            
            repeat_unit_charges.append(float(line.split()[-1]))
    return repeat_unit_charges

In [ ]:

def create_atom_charges_dict(repeat_unit_charges, mol, n):
    
    
    
    mol_with_hydrogens = Chem.AddHs(mol)
    
    
    atom_charges_dict = {}
    
    
    atoms_formal_charged_dict = {}
    
    
    n = int(n)
    
    for repeat_unit in range(1, n + 1):
        for atom in mol_with_hydrogens.GetAtoms():
            atom_idx = atom.GetIdx()  
            atom_symbol = atom.GetSymbol()  
            atom_formal_charge = atom.GetFormalCharge()  
            if atom_symbol == '*':  
                atom_name = f"*{atom_idx}_{repeat_unit}"
            else:
                atom_name = f"{atom_symbol}{atom_idx}_{repeat_unit}"
            
            atom_charge = repeat_unit_charges[atom_idx]
            
            atom_charges_dict[atom_name] = atom_charge
            
            
            if atom_formal_charge != 0:
                
                atoms_formal_charged_dict[atom_name] = atom_formal_charge
            
    
    return atom_charges_dict, atoms_formal_charged_dict

In [ ]:
def create_index_formal_charge_dict(node_list, atoms_formal_charged_dict):
    
    index_formal_charge_dict = {}

    
    for index, node in enumerate(node_list):
        if node in atoms_formal_charged_dict:
            
            index_formal_charge_dict[index] = atoms_formal_charged_dict[node]

    
    return index_formal_charge_dict

In [ ]:

def create_end_group_atom_charges_dict(repeat_unit_charges, mol, n):
    
    
    
    mol_with_hydrogens = Chem.AddHs(mol)
    
    end_group_atom_charges_dict = {}  

    
    virtual_atom_indices = []

    
    for atom in mol_with_hydrogens.GetAtoms():
        if atom.GetSymbol() == '*':  
            virtual_atom_indices.append(atom.GetIdx())

    
    virtual_atom_indices.sort()

    
    if len(virtual_atom_indices) < 2:
        raise ValueError("Not enough virtual atoms to determine end groups.")

    
    beginning_atom_idx = virtual_atom_indices[0]  
    ending_atom_idx = virtual_atom_indices[-1]    

    
    end_group_atom_charges_dict[f"*{beginning_atom_idx}_1"] = repeat_unit_charges[beginning_atom_idx]
    end_group_atom_charges_dict[f"*{ending_atom_idx}_{n}"] = repeat_unit_charges[ending_atom_idx]

    
    return end_group_atom_charges_dict

In [ ]:

bond_list = [Chem.rdchem.BondType.UNSPECIFIED,
             Chem.rdchem.BondType.SINGLE, 
             Chem.rdchem.BondType.DOUBLE,
             Chem.rdchem.BondType.TRIPLE, 
             Chem.rdchem.BondType.QUADRUPLE, 
             Chem.rdchem.BondType.QUINTUPLE,
             Chem.rdchem.BondType.HEXTUPLE, 
             Chem.rdchem.BondType.ONEANDAHALF, 
             Chem.rdchem.BondType.TWOANDAHALF,
             Chem.rdchem.BondType.THREEANDAHALF, 
             Chem.rdchem.BondType.FOURANDAHALF, 
             Chem.rdchem.BondType.FIVEANDAHALF,
             Chem.rdchem.BondType.AROMATIC, 
             Chem.rdchem.BondType.IONIC, 
             Chem.rdchem.BondType.HYDROGEN,
             Chem.rdchem.BondType.THREECENTER,
             Chem.rdchem.BondType.DATIVEONE,
             Chem.rdchem.BondType.DATIVE,
             Chem.rdchem.BondType.DATIVEL, 
             Chem.rdchem.BondType.DATIVER,
             Chem.rdchem.BondType.OTHER,
             Chem.rdchem.BondType.ZERO]


In [ ]:
def all_bonds_info_and_order(bond_list, mol, n):
    
    
    
    mol_with_hydrogens = Chem.AddHs(mol)

    
    all_bonds_info = []

    
    bonds_info_with_order = []

    
    for repeat_unit in range(1, n + 1):
        
        for bond in mol_with_hydrogens.GetBonds():
            
            begin_idx = bond.GetBeginAtomIdx()
            end_idx = bond.GetEndAtomIdx()
            begin_atom_symbol = mol_with_hydrogens.GetAtomWithIdx(begin_idx).GetSymbol()
            end_atom_symbol = mol_with_hydrogens.GetAtomWithIdx(end_idx).GetSymbol()

            
            begin_atom_name = f"{begin_atom_symbol}{begin_idx}_{repeat_unit}"
            end_atom_name = f"{end_atom_symbol}{end_idx}_{repeat_unit}"

            
            bond_order = bond_list.index(bond.GetBondType())

            
            if begin_atom_symbol == '*':
                begin_atom_name = f"*{begin_idx}_{repeat_unit}"
            if end_atom_symbol == '*':
                end_atom_name = f"*{end_idx}_{repeat_unit}"

            
            bonds_info_with_order.append(((begin_atom_name, end_atom_name), bond_order))

            
            all_bonds_info.append((begin_atom_name, end_atom_name))
   
    return all_bonds_info, bonds_info_with_order

In [ ]:

def find_aggregation_sites(mol):
    
    dummy_atoms = [(atom.GetIdx(), atom.GetNeighbors()[0].GetIdx()) for atom in mol.GetAtoms() if atom.GetSymbol() == '*']
    
    assert len(dummy_atoms) == 2, "There should be exactly two aggregation sites."
    
    dummy_atoms.sort(key=lambda x: x[0])
    return dummy_atoms

In [ ]:
def build_polymer_connections(mol, n, aggregation_sites):
    
    a_site, b_site = aggregation_sites 
    a_dummy_atom_idx, b_dummy_atom_idx = a_site[0], b_site[0] 
    a_connected_atom_idx, b_connected_atom_idx = a_site[1], b_site[1] 
    
    
    polymerization_bonds_info_with_order = []
    
    print(a_site, b_site)
    print(a_connected_atom_idx, b_connected_atom_idx)
    
    
    connections = []
    
    
    end_group_connections=[]
    
    a_atom_symbol = mol.GetAtomWithIdx(a_site[1]).GetSymbol()
    b_atom_symbol = mol.GetAtomWithIdx(b_site[1]).GetSymbol()
    
    
    polymerization_bondorder = 1
    
    
    for i in range(2, n+1):
        
        connections.append((f"{b_atom_symbol}{b_connected_atom_idx}_{i-1}", f"{a_atom_symbol}{a_connected_atom_idx}_{i}"))
        
        polymerization_bonds_info_with_order.append(((f"{b_atom_symbol}{b_connected_atom_idx}_{i-1}", f"{a_atom_symbol}{a_connected_atom_idx}_{i}"), polymerization_bondorder))
    
    
    end_group_connections.append((f"*{a_dummy_atom_idx}_1", f"{a_atom_symbol}{a_connected_atom_idx}_1"))  
    end_group_connections.append((f"*{b_dummy_atom_idx}_{n}", f"{b_atom_symbol}{b_connected_atom_idx}_{n}"))  
    
    
    polymerization_bonds_info_with_order.append(((f"{a_atom_symbol}{a_connected_atom_idx}_1", f"*{a_dummy_atom_idx}_1"), polymerization_bondorder))
    polymerization_bonds_info_with_order.append(((f"{b_atom_symbol}{b_connected_atom_idx}_{n}", f"*{b_dummy_atom_idx}_{n}"), polymerization_bondorder))
    
    return connections, polymerization_bonds_info_with_order, end_group_connections

In [ ]:

def create_undirected_graph(node_list, final_polymer_connection_list):
    
    
    G = nx.Graph()

    
    G.add_nodes_from(node_list)

    
    G.add_edges_from(final_polymer_connection_list)

    
    pos = nx.kamada_kawai_layout(G)

    
    plt.figure(figsize=(12, 8))
    nx.draw(G, pos, with_labels=True, node_color='lightblue', edge_color='gray', node_size=200, font_size=10)
    plt.title("Polymer Chain Graph")
    plt.show()

In [ ]:
def create_polymer_adjacency_matrix(node_list, final_polymer_bonds_info_with_order):
    
    polymer_adjacency_matrix = np.zeros((len(node_list), len(node_list)))
    
    
    node_index = {node: idx for idx, node in enumerate(node_list)}
    
    
    for bond, order in final_polymer_bonds_info_with_order:
        node1, node2 = bond
        idx1, idx2 = node_index[node1], node_index[node2]
        polymer_adjacency_matrix[idx1][idx2] = order
        polymer_adjacency_matrix[idx2][idx1] = order  
    return polymer_adjacency_matrix

In [ ]:

def create_heatmap_polymer_adjacency_matrix(adjacency_matrix, node_list, title='Polymer Adjacency Matrix Heatmap', annot=False, fontsize=5, xlabelsize=5, ylabelsize=5):
    
    plt.figure(figsize=(8, 6))
    
    sns.heatmap(adjacency_matrix, annot=annot, cmap='Greys', xticklabels=node_list, yticklabels=node_list, fmt='g')
    
    plt.title(title, fontsize=fontsize)
    
    plt.xticks(fontsize=xlabelsize)
    plt.yticks(fontsize=ylabelsize)
    
    plt.show()

In [ ]:

def create_element_node_list(node_list):
    
    element_node_list = []

    
    pattern = re.compile(r"([A-Z][a-z]*)\d*_*\d*")

    
    for node in node_list:
        if node.startswith("*"):  
            element_node_list.append("*")
        else:
            match = pattern.match(node)
            if match:
                
                element_name = match.group(1)
                element_node_list.append(element_name)
            else:
                
                element_node_list.append(node)
    
    return element_node_list

In [ ]:

def graph2mol(atoms, ad_martrix, index_formal_charge_dict):
    
    new_mol = Chem.RWMol()
    
    atom_index = []
    
    
    for atom_number in range(len(atoms)):
        atom = Chem.Atom(atoms[atom_number])  
        molecular_index = new_mol.AddAtom(atom)  
        atom_index.append(molecular_index)  
    
    
    for index_x, row_vector in enumerate(ad_martrix):
        
        for index_y, bond in enumerate(row_vector):
            
            bond = int(bond)
            
            
            if index_y <= index_x:
                continue
            
            if bond == 0:
                continue
            
            else:
                new_mol.AddBond(atom_index[index_x],
                                atom_index[index_y], 
                                bond_list[bond])  
  
    
    for atom_index, formal_charge in index_formal_charge_dict.items():
        
        atom = new_mol.GetAtomWithIdx(atom_index)

        
        
        neighbors = atom.GetNeighbors()
        for neighbor in neighbors:
            
            if neighbor.GetSymbol() == 'H':
                
                editable_mol.RemoveAtom(neighbor.GetIdx())

        
        atom.SetFormalCharge(formal_charge)

    
    updated_polymer_mol = new_mol.GetMol()

    
    Chem.SanitizeMol(new_mol)
    
    
    new_mol = new_mol.GetMol()
    
    
    return new_mol


In [ ]:


def cap_with_hydrogen(mol):
    
    Chem.SanitizeMol(mol)
    
    mol_with_h = AddHs(mol)

    
    terminal_indices = [atom.GetIdx() for atom in mol_with_h.GetAtoms() if atom.GetSymbol() == '*']
    print("Terminal indices:", terminal_indices)

    
    non_terminal_indices = [atom.GetIdx() for atom in mol_with_h.GetAtoms() if atom.GetSymbol() != '*']
    print("Non-terminal indices:", non_terminal_indices)

    
    rw_mol = RWMol(mol_with_h)

    
    for idx in terminal_indices:
        rw_mol.ReplaceAtom(idx, Chem.Atom(1))  
        rw_mol.GetAtomWithIdx(idx).SetNoImplicit(False)  

    
    rw_mol.UpdatePropertyCache(strict=False)

    
    capped_mol = rw_mol.GetMol()
    
    return capped_mol

In [ ]:


def calculate_charge_and_multiplicity(mol):
    
    mol = Chem.AddHs(mol)
    
    
    charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
    
    
    multiplicity = sum([atom.GetNumRadicalElectrons() for atom in mol.GetAtoms()]) + 1
    
    return charge, multiplicity

In [ ]:
def create_polymer_Gaussian_inputfile(polymer_capped_mol, polymer_name, nproc, mem):
    
    
    params = AllChem.ETKDGv3()
    embed_status = AllChem.EmbedMolecule(polymer_capped_mol, params)

    if embed_status == 0:
        print("3D conformation generated using ETKDGv3.")
    else:
        
        print("ETKDGv3 failed. Attempting distance geometry.")
        embed_status = AllChem.EmbedMolecule(polymer_capped_mol, useRandomCoords=True)
        if embed_status != 0:
            
            print("Failed to generate 3D conformation with distance geometry.")
            return None
    
    
    optimization_status = AllChem.MMFFOptimizeMolecule(polymer_capped_mol, mmffVariant="MMFF94")
    if optimization_status >= 0:
        print("Optimization with MMFF94 successful.")
    else:
        
        print("Optimization with MMFF94 failed. Attempting optimization with UFF.")
        AllChem.UFFOptimizeMolecule(polymer_capped_mol)

    
    obConversion = ob.OBConversion()
    
    
    obConversion.SetInAndOutFormats("sdf", "mol2")
    
    
    ob_mol = ob.OBMol()
    
    
    ob_mol = ob.OBMol()
    notatend = obConversion.ReadString(ob_mol, Chem.MolToMolBlock(polymer_capped_mol))
    
    
    obConversion.WriteFile(ob_mol, f"polymer_{polymer_name}.mol2")
    
    
    obConversion.SetOutFormat("pdb")
    
    
    obConversion.WriteFile(ob_mol, f"polymer_{polymer_name}.pdb")
    
    
    xyz = Chem.MolToXYZBlock(polymer_capped_mol)
    
    
    filename_xyz = f"polymer_{polymer_name}.xyz"

    
    with open(filename_xyz, 'w') as file:
        file.write(xyz)

In [ ]:


def clean_mol2_files():
    
    unity_atom_attr_pattern = re.compile(r'@<TRIPOS>UNITY_ATOM_ATTR')
    bond_pattern = re.compile(r'@<TRIPOS>BOND')

    
    for filename in os.listdir('.'):
        
        if filename.startswith('polymer_') and filename.endswith('.mol2'):
            with open(filename, 'r') as file:
                lines = file.readlines()

            
            in_unity_atom_attr_block = False
            
            new_lines = []

            
            for line in lines:
                
                if unity_atom_attr_pattern.search(line):
                    in_unity_atom_attr_block = True  
                    print('find @<TRIPOS>UNITY_ATOM_ATTR')
                    continue  
                
                
                if in_unity_atom_attr_block and bond_pattern.search(line):
                    in_unity_atom_attr_block = False  
                    print('find @<TRIPOS>UNITY_ATOM_ATTR and @<TRIPOS>BOND')

                
                if not in_unity_atom_attr_block:
                    new_lines.append(line)

            
            if in_unity_atom_attr_block or (new_lines != lines):
                with open(filename, 'w') as file:
                    file.writelines(new_lines)


In [ ]:
clean_mol2_files()

In [ ]:

def polymer_chg(polymer_name, polymer_charges_list):
    
    
    df_charges = pd.DataFrame(polymer_charges_list, columns=['Charge'])

    
    
    df_xyz = pd.read_csv(f"polymer_{polymer_name}.xyz", sep='\s+', names=['Element', 'X', 'Y', 'Z'])

    
    df_xyz = df_xyz.drop(df_xyz.index[0])

    
    df_xyz = df_xyz.reset_index(drop=True)

    
    df_combined = pd.concat([df_xyz, df_charges], axis=1)

    
    output_filename = f"polymer_{polymer_name}.chg"

    
    df_combined.to_csv(output_filename, sep=' ', index=False, header=False)

In [ ]:



from typing import Dict, List  
from rdkit import Chem        
import os                     
import json                   
import re                     

def _sanitize_filename(name: str) -> str:
    
    
    return re.sub(r'[\\/:\*\?"<>\|]+', '_', name)

def save_polymer_atom_lists(polymer_repeat_unit: Dict[str, Chem.Mol], output_dir: str = ".") -> Dict[str, str]:
    
    
    if not isinstance(polymer_repeat_unit, dict):
        raise TypeError("Public message removed for release.")

    
    os.makedirs(output_dir, exist_ok=True)

    
    name_to_path: Dict[str, str] = {}

    
    for name, mol in polymer_repeat_unit.items():
        
        if not isinstance(mol, Chem.Mol):
            raise TypeError("Public message removed for release.")

        
        mol_with_hydrogens = Chem.AddHs(mol)

        
        polymer_atom_list: List[str] = []

        
        for atom_idx, atom in enumerate(mol_with_hydrogens.GetAtoms()):
            
            atom_symbol = atom.GetSymbol()

            
            if atom_symbol == '*':
                
                
                atom_name = f"*{atom_idx}" 
            else:
                
                
                atom_name = f"{atom_symbol}{atom_idx}"

            
            polymer_atom_list.append(atom_name)

        
        safe_name = _sanitize_filename(name)
        file_name = f"{safe_name}_atom_list.json"

        
        file_path = os.path.join(output_dir, file_name)

        
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(polymer_atom_list, f, ensure_ascii=False, indent=2)

        
        name_to_path[name] = os.path.abspath(file_path)

    
    return name_to_path


def load_polymer_atom_list(file_path: str) -> List[str]:
    
    
    if not os.path.isfile(file_path):
        raise FileNotFoundError("Public message removed for release.")

    
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    
    if not isinstance(data, list):
        raise ValueError("Public message removed for release.")

    
    if not all(isinstance(x, str) for x in data):
        raise ValueError("Public message removed for release.")

    
    return data

In [ ]:

def processing(polymer_name, smiles, n, nproc=20, mem=64):
    
    
    chg_filename = f"{polymer_name}.chg"
    
    
    mol = Chem.MolFromSmiles(smiles)
    
    
    repeat_unit_charges = repeat_unit_charge_list(chg_filename)
    print(repeat_unit_charges)
    
    end_group_atom_charges_dict = create_end_group_atom_charges_dict(repeat_unit_charges, mol, n)
    
    
    atom_charges_dict, atoms_formal_charged_dict = create_atom_charges_dict(repeat_unit_charges, mol, n)
    print(atom_charges_dict)
    
    
    del_end_group_atom_charges_dict = {k: v for k, v in atom_charges_dict.items() if not k.startswith("*") or k in end_group_atom_charges_dict}

    
    final_atom_charges_dict = {**del_end_group_atom_charges_dict, **end_group_atom_charges_dict}

    
    polymer_charges_list = list(final_atom_charges_dict.values())

    
    all_bonds_info, bonds_info_with_order = all_bonds_info_and_order(bond_list, mol, n)

    
    aggregation_sites = find_aggregation_sites(mol)

    
    polymer_connections, polymer_bonds_info_with_order, end_group_connections = build_polymer_connections(mol, n, aggregation_sites)
    print(polymer_connections)
    
    
    del_end_group_connection_list = [bond for bond in all_bonds_info if "*" not in bond[0] and "*" not in bond[1]]

    
    final_polymer_connection_list = del_end_group_connection_list + polymer_connections + end_group_connections

    print(final_polymer_connection_list)

    
    
    node_list = list(final_atom_charges_dict.keys())
    
    print(node_list)

    
    index_formal_charge_dict = create_index_formal_charge_dict(node_list, atoms_formal_charged_dict)
    
    
    create_undirected_graph(node_list, final_polymer_connection_list)

    
    del_end_group_bonds_info_with_order = [bond for bond in bonds_info_with_order if "*" not in bond[0][0] and "*" not in bond[0][1]]

    
    final_polymer_bonds_info_with_order = del_end_group_bonds_info_with_order + polymer_bonds_info_with_order

    
    adjacency_matrix = create_polymer_adjacency_matrix(node_list, final_polymer_bonds_info_with_order)

    
    create_heatmap_polymer_adjacency_matrix(adjacency_matrix, node_list)

    
    element_node_list = create_element_node_list(node_list)

    
    polymer_mol = graph2mol(element_node_list, adjacency_matrix, index_formal_charge_dict)
    
    
    Chem.SanitizeMol(polymer_mol)

    
    polymer_capped_mol = cap_with_hydrogen(polymer_mol)

    
    create_polymer_Gaussian_inputfile(polymer_capped_mol, polymer_name, nproc, mem)

    
    clean_mol2_files()
    
    
    polymer_chg(polymer_name, polymer_charges_list)

    
    
    df = pd.read_excel('System_homopolymer.xlsx')

    
    polymer_smiles = {} 
    polymer_repeat_unit = {} 

    
    for index, row in df.iterrows():
        
        polymer_smiles[row['Name']] = row['SMILES']

    
    for polymer_name, smiles in polymer_smiles.items():
        
        mol = Chem.MolFromSmiles(smiles)
        
        mol = Chem.AddHs(mol)
        
        polymer_repeat_unit[polymer_name] = mol

    paths = save_polymer_atom_lists(polymer_repeat_unit, output_dir='.')

    
    with open("polymer_node_list.json", "w", encoding="utf-8") as f:
        json.dump(node_list, f, ensure_ascii=False, indent=2)

    
    print("-----------------------------------------polymer_mol----------------------------------------------------")
    img = Draw.MolToImage(polymer_mol, size=(300, 300))
    display(img)

In [ ]:

df = pd.read_excel('System_homopolymer.xlsx')


polymer_name = df['Name'].astype(str).tolist()


for name in polymer_name:
    
    if name in df['Name'].values:
        print(f"--------{name}-------{name}----------{name}--------{name}---------{name}-------{name}----------{name}--------{name}--------")
        
        smiles = df.loc[df['Name'] == name, 'SMILES'].values[0]
        n = int(df.loc[df['Name'] == name, 'repeating unit'].values[0])
        processing(name, smiles, n)
        print(f"--------{name}-------{name}----------{name}--------{name}---------{name}-------{name}----------{name}--------{name}--------")